# RAG PDF Chatbot Experiement Notebook
---

**Register your UV venv environment with Jupyter**
```
uv run python -m ipykernel install --user --name rag-env --display-name "Python (RAG)"
```

---

In [ ]:
from dotenv import load_dotenv
load_dotenv("../.env")

import os
print("Gemini API Key Loaded: ", bool(os.getenv("GOOGLE_API_KEY")))


## Configuration
---

In [ ]:
os.getcwd()


In [ ]:
DOCS_BASE = "../data"
DOC_PATHS = [
  "attention_is_all_you_need_paper.pdf"
]


## Load Document
---

In [ ]:
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader


def load_documents(doc_paths: list[str] | None = None) -> list[Document]:
  paths = doc_paths or DOC_PATHS
  docs: list[Document] = []

  for path in paths:
    source = f"{DOCS_BASE}/{path}"
    pdf_loader = PyPDFLoader(source)
    pdf_extracted = pdf_loader.load() # extract text from pdf
    # pdf_extracted is list of documents i.e one page is one Document
    docs.extend(pdf_extracted)

  return docs


In [ ]:
docs = load_documents()
print(f"Loaded {len(docs)} documentation pages.")


In [ ]:
docs[0]


In [ ]:
print("Page 7 first 500 characters: \n")
print(docs[6].page_content[:500])


In [ ]:
total_chars = sum(len(doc.page_content) for doc in docs)
print(f"Total characters: {total_chars}")


**Rule of thumb: 1 token ≈ 4 English characters**
- So 32 thousand characters is around 8000 tokens

## Splitting or Chunking the Document
---

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


def chunk_documents(raw_docs: list[Document]):
  text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=100)
  chunks = text_splitter.split_documents(raw_docs)
  return chunks


In [ ]:
chunks = chunk_documents(docs)


In [ ]:
chunks[0]


In [ ]:
print(f"Split documentation into {len(chunks)} chunks.")


In [ ]:
print("Chunk 8 first 100 characters: \n")
print(chunks[7].page_content[:100])


In [ ]:
total_chars = sum(len(chunk.page_content) for chunk in chunks)
print(f"Total characters: {total_chars}")


**Notice the total characters increased after chunking**
- Becuase of chunk overlap

## Embedding Model
---

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings


embedding_model = GoogleGenerativeAIEmbeddings(model='models/gemini-embedding-001') # 100 RPM, 100 RPD, 30K TPM


In [ ]:
embedding_model


## Vector Store(run only once)
---

In [ ]:
from langchain_chroma import Chroma


vector_store = Chroma(
  collection_name="first_collection",
  embedding_function=embedding_model,
  persist_directory="../chroma_db"
)


In [ ]:
# Embedding in batches
# Cuz i'm using free tier: 100 RPM i.e 100 chunks per minute limit
# I have a total of 168 chunks

import time

batch_size = 100

for i in range(0, len(chunks), batch_size):
    batch = chunks[i:i + batch_size]

    vector_store.add_documents(batch)

    print(f"Added {len(batch)} chunks")

    if i + batch_size < len(chunks):
        print("Sleeping 60 secs...")
        time.sleep(60)

print("All Chunks indexed")


**NOTE: do not rerun above as it'll repeat and duplicate the document embeddings**

## Agent
---